In [1]:
library(tidyverse)
library(dplyr)

── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [7]:
nhl_players <- readRDS ("Predict NHL Breakout 26-27/nhl_players.rds")

In [42]:
nhl_players <- nhl_players |>
    filter(situation == "all")

nrow(nhl_players)

[1] 10115

In [9]:
nhl_players <- nhl_players |>
    select (where(~ !all(is.na(.))))

In [44]:
invisible(colnames(nhl_players))

In [46]:
nhl_clean <- nhl_players |>
    transmute(
        player_id = playerId,
        player_name = name,
        season_start = season,
        team,
        position,
        games_played, 

        total_minutes = icetime/ 60,
        average_minutes = total_minutes / games_played, 

        goals = I_F_goals, 
        primary_assists = I_F_primaryAssists, 
        secondary_assists = I_F_secondaryAssists,
        assists = primary_assists + secondary_assists,
        points = I_F_points,
    
        points_per_game = points / games_played,
        points_per_82 = points_per_game * 82
  )
        

In [47]:
nrow(nhl_clean)

[1] 10115

In [17]:
ncol(nhl_clean)

[1] 15

In [48]:
print(
    head(nhl_clean, 20),
    n = 20
    )

# A tibble: 20 × 15
   player_id player_name  season_start team  position games_played total_minutes
       <dbl> <chr>               <dbl> <chr> <chr>           <dbl>         <dbl>
 1   8469459 Mikko Koivu          2015 MIN   C                  82        1634. 
 2   8477845 Trevor van …         2015 CHI   D                  82        1639. 
 3   8470724 Kyle Quincey         2015 DET   D                  47         929. 
 4   8476887 Filip Forsb…         2015 NSH   L                  82        1562. 
 5   8466285 Brian Campb…         2015 FLA   D                  82        1827. 
 6   8474052 Chris Terry          2015 CAR   L                  68         767. 
 7   8478393 Kyle Baun            2015 CHI   R                   2          18.1
 8   8474037 James van R…         2015 TOR   L                  40         711. 
 9   8476449 Scott Harri…         2015 TOR   D                  15         196. 
10   8475727 Ryan Spooner         2015 BOS   C                  80        1211. 
11   847

In [49]:
player_bios <- read_csv(
    "Predict NHL breakout 26-27/player_bios.csv",
    show_col_types = FALSE
    )
names(player_bios)
head(player_bios)

[1] "playerId"        "name"            "position"        "team"           
 [5] "birthDate"       "weight"          "height"          "nationality"    
 [9] "shootsCatches"   "primaryNumber"   "primaryPosition"

playerId,name,position,team,birthDate,weight,height,nationality,shootsCatches,primaryNumber,primaryPosition
<dbl>,<chr>,<chr>,<chr>,<date>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>
8478421,A.J. Greer,L,FLA,1996-12-14,210,"6' 3""",CAN,L,24,L
8477180,Aaron Dell,G,SJS,1989-05-04,205,"6' 0""",CAN,L,30,G
8465992,Aaron Downey,R,DET,1974-08-27,215,"6' 1""",CAN,R,44,R
8477932,Aaron Ekblad,D,FLA,1996-02-07,220,"6' 4""",CAN,R,5,D
8471451,Aaron Gagnon,C,WPG,1986-04-24,186,"5' 11""",CAN,R,21,C
8469534,Aaron Johnson,D,BOS,1983-04-30,204,"6' 1""",CAN,L,72,D


In [50]:
player_bios_clean <- player_bios |>
    transmute(
        player_id = playerId,
        birth_date = as.Date(birthDate)
        )

In [51]:
nhl_clean <- nhl_clean |>
    left_join(
        player_bios_clean,
        by = "player_id"
        )

In [53]:
head(nhl_clean, 20)

player_id,player_name,season_start,team,position,games_played,total_minutes,average_minutes,goals,primary_assists,secondary_assists,assists,points,points_per_game,points_per_82,birth_date
<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<date>
8469459,Mikko Koivu,2015,MIN,C,82,1634.18333,19.929065,17,18,21,39,56,0.68292683,56.000000,1983-03-12
8477845,Trevor van Riemsdyk,2015,CHI,D,82,1638.66667,19.983740,3,7,4,11,14,0.17073171,14.000000,1991-07-24
8470724,Kyle Quincey,2015,DET,D,47,928.88333,19.763475,4,0,7,7,11,0.23404255,19.191489,1985-08-12
8476887,Filip Forsberg,2015,NSH,L,82,1562.31667,19.052642,33,14,17,31,64,0.78048780,64.000000,1994-08-13
8466285,Brian Campbell,2015,FLA,D,82,1826.78333,22.277846,6,12,13,25,31,0.37804878,31.000000,1979-05-23
8474052,Chris Terry,2015,CAR,L,68,766.68333,11.274755,8,2,1,3,11,0.16176471,13.264706,1989-04-07
8478393,Kyle Baun,2015,CHI,R,2,18.10000,9.050000,0,0,0,0,0,0.00000000,0.000000,NA
8474037,James van Riemsdyk,2015,TOR,L,40,710.51667,17.762917,14,10,5,15,29,0.72500000,59.450000,1989-05-04
8476449,Scott Harrington,2015,TOR,D,15,196.41667,13.094444,0,0,1,1,1,0.06666667,5.466667,1993-03-10


In [55]:
nhl_clean <- nhl_clean |>
    mutate(
        season_date = as.Date (
            paste0(season_start, "-10-01")
            ),

        age = as.numeric (
            difftime(
                season_date,
                birth_date,
                units = "days"
                )
            ) / 365.25
        )
head(nhl_clean, 20)

player_id,player_name,season_start,team,position,games_played,total_minutes,average_minutes,goals,primary_assists,secondary_assists,assists,points,points_per_game,points_per_82,birth_date,season_date,age
<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<date>,<date>,<dbl>
8469459,Mikko Koivu,2015,MIN,C,82,1634.18333,19.929065,17,18,21,39,56,0.68292683,56.000000,1983-03-12,2015-10-01,32.55578
8477845,Trevor van Riemsdyk,2015,CHI,D,82,1638.66667,19.983740,3,7,4,11,14,0.17073171,14.000000,1991-07-24,2015-10-01,24.18891
8470724,Kyle Quincey,2015,DET,D,47,928.88333,19.763475,4,0,7,7,11,0.23404255,19.191489,1985-08-12,2015-10-01,30.13552
8476887,Filip Forsberg,2015,NSH,L,82,1562.31667,19.052642,33,14,17,31,64,0.78048780,64.000000,1994-08-13,2015-10-01,21.13347
8466285,Brian Campbell,2015,FLA,D,82,1826.78333,22.277846,6,12,13,25,31,0.37804878,31.000000,1979-05-23,2015-10-01,36.35866
8474052,Chris Terry,2015,CAR,L,68,766.68333,11.274755,8,2,1,3,11,0.16176471,13.264706,1989-04-07,2015-10-01,26.48323
8478393,Kyle Baun,2015,CHI,R,2,18.10000,9.050000,0,0,0,0,0,0.00000000,0.000000,NA,2015-10-01,NA
8474037,James van Riemsdyk,2015,TOR,L,40,710.51667,17.762917,14,10,5,15,29,0.72500000,59.450000,1989-05-04,2015-10-01,26.40931
8476449,Scott Harrington,2015,TOR,D,15,196.41667,13.094444,0,0,1,1,1,0.06666667,5.466667,1993-03-10,2015-10-01,22.55989


In [56]:
nrow(head(nhl_clean, 20))

[1] 20

In [57]:
nhl_clean |>
    filter(is.na(age))|>
    distinct(player_id, player_name)

player_id,player_name
<dbl>,<chr>
8478393,Kyle Baun
8475247,Michael Latta
8469490,Tim Jackman
8470230,Daniel Paille
8471996,Bracken Kearns
8467514,Pavel Datsyuk
8476404,Andy Andreoff
8478584,Vojtech Mozik
8470852,Matt Moulson


In [58]:
nhl_clean |>
    filter(is.na(age))|>
    select(
        player_id,
        player_name,
        season_start,
        team
        )

player_id,player_name,season_start,team
<dbl>,<chr>,<dbl>,<chr>
8478393,Kyle Baun,2015,CHI
8475247,Michael Latta,2015,WSH
8469490,Tim Jackman,2015,ANA
8470230,Daniel Paille,2015,NYR
8471996,Bracken Kearns,2015,NYI
8467514,Pavel Datsyuk,2015,DET
8476404,Andy Andreoff,2015,L.A
8478584,Vojtech Mozik,2015,N.J
8470852,Matt Moulson,2015,BUF


In [59]:
mean(!is.na(nhl_clean$age))*100

[1] 98.58626

In [60]:
nhl_model_ready <- nhl_clean |>
    filter(!is.na(age))

#filtering out players whose age could not be matched with a player id on moneypuck

In [61]:
sum(is.na(nhl_model_ready$age))

[1] 0

In [62]:
summary(nhl_model_ready$age)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  18.05   23.42   26.14   26.67   29.51   45.63 

In [63]:
nhl_model_ready |>
    arrange(age)|>
    select(player_name, season_start, age) |>
    slice_head(n = 10)

player_name,season_start,age
<chr>,<dbl>,<dbl>
Alex Formenton,2017,18.05065
Filip Chytil,2017,18.07255
Matthew Schaefer,2025,18.07255
Jett Luchanko,2024,18.11362
Quinton Byfield,2020,18.11910
Clayton Keller,2016,18.17659
Connor Bedard,2023,18.20671
Jesperi Kotkaniemi,2018,18.23682
Mikhail Sergachev,2016,18.26968


In [66]:
nhl_outcomes <- nhl_model_ready |>
    arrange(player_id, season_start) |>
    group_by(player_id) |>
    mutate (
        next_season = lead(season_start),
        next_games_played = lead(games_played),
        next_points_per_82 = lead(points_per_82)
        ) |>
ungroup()

##starts with the whole dataset not just active cases

In [68]:
nhl_outcomes <- nhl_outcomes |>
    filter(next_season == season_start + 1) |>
##removes cases where players missed a season before returning

In [70]:
nhl_outcomes <- nhl_outcomes |>
    filter(
        games_played >= 30,
        next_games_played >= 30
        )

##minimum 30 games per season requirement in order for stats to be relevant

In [72]:
nhl_outcomes <- nhl_outcomes |>
    mutate(
        points_increase = 
        next_points_per_82 - points_per_82,

        percentage_increase = if_else(
            points_per_82 > 0,
            points_increase / points_per_82,
            NA_real_
            ),
        breakout = if_else (
            points_increase >= 15 &
            percentage_increase >= 0.25,
            1L,
            0L
            )
        ) |>
    filter(!is.na(breakout))

#define a breakout by putting it into a binary system
# 1  = breakout
# 0 = no breakout

In [73]:
nhl_outcomes |>
    count(breakout) |>
    mutate(percentage = n / sum(n) * 100)

breakout,n,percentage
<int>,<int>,<dbl>
0,4670,89.70419
1,536,10.29581


In [74]:
training_data <- nhl_outcomes |>
  filter(season_start <= 2022)

validation_data <- nhl_outcomes |>
  filter(season_start == 2023)

testing_data <- nhl_outcomes |>
  filter(season_start == 2024)

prediction_data <- nhl_model_ready |>
  filter(season_start == 2025)

In [75]:
range(training_data$season_start)
unique(validation_data$season_start)
unique(testing_data$season_start)
unique(prediction_data$season_start)

[1] 2015 2022

[1] 2023

[1] 2024

[1] 2025

In [76]:
tibble(
  dataset = c(
    "Training",
    "Validation",
    "Testing",
    "Prediction"
  ),
  rows = c(
    nrow(training_data),
    nrow(validation_data),
    nrow(testing_data),
    nrow(prediction_data)
  )
)

#looking to build a model with past information, validate on a later season
#test on a new season and generate real predictions for 2026/2027 season

dataset,rows
<chr>,<int>
Training,4112
Validation,552
Testing,542
Prediction,904


In [77]:
saveRDS(nhl_outcomes, 
        "Predict NHL Breakout 26-27/nhl_outcomes.rds"
        )